# Storage and partitioning

The half of BigQuery you can model on your laptop.

BigQuery organises data into **datasets**, and datasets hold **tables** and **views**. The
tables live in **Colossus**, Google’s distributed storage system, in **Capacitor** — a
compressed columnar format — and **Dremel** executes your SQL across **slots**.

The memory trick for the whole architecture:

```text
S → C → C → D → S
Serverless → Colossus → Capacitor → Dremel → Slots
```

Everything in this notebook follows from one idea: **BigQuery bills for bytes scanned**, so
every layout decision is a cost decision.


## 0. Setup

Put the course’s pure core modules on the path. No installation and no cloud SDK.


In [ ]:
import sys
from pathlib import Path

# The course's pure core modules. No installation and no cloud SDK — that is
# the whole point of the split.
COURSE_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd().parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))

from cost_model import (
    BIGQUERY_ONDEMAND_PER_TB,
    BIGQUERY_SLOT_HOUR,
    compare_architectures,
    estimate,
    format_comparison,
    monthly_runway,
)
from pipeline_logic import (
    aggregate_events,
    bucket_by_window,
    dedup_key,
    deduplicate,
    estimated_scan_bytes,
    partition_for_date,
    partition_for_event,
)

print("core modules on the path:", (COURSE_ROOT / "project" / "core").exists())


## 1. Partition decorators

BigQuery partitions a table by a date column, and names each partition with a **decorator** —
the date with the dashes removed. The course’s helper produces the same convention.


In [ ]:
print("logical date ->", partition_for_date("2026-01-15"))
print("event time  ->", partition_for_event("2026-01-15T09:30:00+00:00"))
print("decorator   ->", partition_for_date("2026-01-15"))


## 2. How much does a table actually weigh?

BigQuery is **columnar**: it stores data in columns, in blocks. A query that reads two columns
reads two columns’ worth of bytes — the rest are never opened. That is the mechanism behind
every cost optimisation BigQuery has.


In [ ]:
rows = [
    {
        "event_id": f"e{i:08d}",
        "user_id": f"u{i:06d}",
        "event_dt": "2026-01-15",
        "region": ("AMER", "EMEA", "APAC", "LATAM")[i % 4],
        "event_type": ("order", "refund", "page_view", "signup")[i % 4],
        "amount_eur": 10.0 + (i % 97),
    }
    for i in range(4800)
]

full = estimated_scan_bytes(rows, partitioned=False)
pruned = estimated_scan_bytes(rows, partitioned=True, partition_prune_ratio=0.01)

print(f"rows:            {len(rows):,}")
print(f"unpartitioned:   {full:,} bytes")
print(f"pruned to 1%:    {pruned:,} bytes")
print(f"reduction:       {full / pruned:.0f}x")


## 3. The number to remember

A 10 TB table, scanned whole versus pruned to one partition. **$62.50 versus $0.63** — and the
saving came from the *table layout*, not from rewriting the query.


In [ ]:
whole = estimate("bigquery", scanned_tb=10.0, apply_free_tier=False).total
pruned = estimate("bigquery", scanned_tb=0.1, apply_free_tier=False).total

print(f"unpartitioned: ${whole:.2f}")
print(f"partitioned:   ${pruned:.2f}")
print(f"saving:        ${whole - pruned:.2f}  ({whole / pruned:.0f}x)")


## 4. The free tier trap

`estimate()` applies the monthly free tier by default — 1 TB of scanned bytes. A 1 TB scan
therefore prices at $0.00, which is true and also completely misleading as a price list.

**Always pass `apply_free_tier=False` when you want the price list rather than the bill.**


In [ ]:
print("1 TB, free tier ON :", estimate("bigquery", scanned_tb=1.0).total)
print("1 TB, free tier OFF:", estimate("bigquery", scanned_tb=1.0, apply_free_tier=False).total)
print()
print("per-TB price:", BIGQUERY_ONDEMAND_PER_TB)
print("slot-hour price:", BIGQUERY_SLOT_HOUR)


## 5. Clustering only pays when you filter

Clustering sorts data within a partition so that a **filter** on the clustered column can skip
blocks. A query that aggregates over *every* value of that column gets nothing from it.

This is the most commonly misunderstood point in BigQuery performance work, and it is worth
being able to explain in one sentence.


In [ ]:
# A table clustered by region.
by_region = {"AMER": 0, "EMEA": 0, "APAC": 0, "LATAM": 0}
for row in rows:
    by_region[row["region"]] += 1

print("rows per region:", by_region)
print()
print("Clustering helps when the query says: WHERE region = 'AMER'")
print("Clustering does NOT help when the query says: GROUP BY region")
print()
print("Query 1 (selective filter) reads the AMER blocks only.")
print("Query 2 (aggregate all) reads every block in the partition either way.")


## 6. Idempotency keys, derived from data

A retry must produce the **same** identity, so the duplicate collapses. If the key included a
load timestamp, the retry would produce a *different* key and both rows would survive.

This is the same lesson as Pub/Sub at-least-once delivery, and it is the reason
`dedup_key` takes an event id and a partition — not a clock reading.


In [ ]:
batch = [
    {"event_id": "evt-0001", "event_dt": "2026-01-15", "amount_eur": 10.0},
    {"event_id": "evt-0002", "event_dt": "2026-01-15", "amount_eur": 20.0},
    {"event_id": "evt-0001", "event_dt": "2026-01-15", "amount_eur": 10.0},
]

print("idempotency key:", dedup_key("evt-0001", "20260115"))
print(f"in:  {len(batch)} rows")
print(f"out: {len(deduplicate(batch, partition='20260115'))} rows")


## 7. Windows and aggregation

Windowing derives the bucket from the **event timestamp**, never from arrival time. Get that
wrong and every event collapses into one window — and the totals stay right, which is exactly
why the bug is easy to miss.

Note which field the windowing reads: `event_dt`, and it must be a **full timestamp**. A date
alone puts every event in the same window, because there is no hour to separate them.


In [ ]:
# A day of events, one timestamp per hour.
windowed = [
    {**row, "event_dt": f"2026-01-15T{(i % 24):02d}:00:00+00:00"}
    for i, row in enumerate(rows)
]

buckets = bucket_by_window(windowed)
print(f"hourly windows: {len(buckets)}")
print(f"first window:   {min(buckets)}")
print(f"last window:    {max(buckets)}")
print()
summary = aggregate_events(rows)
for region in sorted(summary):
    stats = summary[region]
    print(f"{region:<6} {int(stats['event_count']):>5} events  ${stats['revenue_eur']:>12,.2f}")


## What you should be able to say now

Without notes:

1. The five-component chain, and the memory trick.
2. Why a 10 TB table costs **$62.50** to scan and **$0.63** when partitioned.
3. Why clustering helps some queries and not others.
4. Why an idempotency key must come from the data.
5. Why `apply_free_tier=False` when you are quoting a price.
